In [1]:
from pathlib import Path
import os
import re
import time
import html
import unicodedata

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
print("Environment ready")

Environment ready


In [2]:
# Colab default: put public_train.csv and public_test.csv in /content/drive/MyDrive/UD_26
# Local default: this repository keeps the files in ./mid

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

DATA_DIR_CANDIDATES = []
if os.environ.get("DATA_DIR"):
    DATA_DIR_CANDIDATES.append(Path(os.environ["DATA_DIR"]))

cwd = Path.cwd()
DATA_DIR_CANDIDATES.extend([
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/MyDrive/UD_26/mid"),
    cwd,
    cwd / "mid",
    cwd / "data",
    cwd.parent,
    cwd.parent / "mid",
])

# Remove duplicates while preserving order.
seen = set()
DATA_DIR_CANDIDATES = [p for p in DATA_DIR_CANDIDATES if not (str(p) in seen or seen.add(str(p)))]

DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    if (candidate / "public_train.csv").exists() and (candidate / "public_test.csv").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    checked = "\n".join(str(p) for p in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(
        "Could not find public_train.csv and public_test.csv.\n"
        "Colab: run this after Drive is mounted and place the files in /content/drive/MyDrive/UD_26 or /content/drive/MyDrive/UD_26/mid.\n"
        "Local: run from the UD_26 repository root or from the mid folder, or set DATA_DIR to the folder containing the CSV files.\n"
        f"Current working directory: {cwd}\n"
        f"Checked paths:\n{checked}"
    )

TRAIN_PATH = DATA_DIR / "public_train.csv"
TEST_PATH = DATA_DIR / "public_test.csv"
OUTPUT_DIR = DATA_DIR
MODEL_DIR = OUTPUT_DIR / "model" / "Best_CV_BinaryCount_Stacking"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = MODEL_DIR / "final_pipeline.pkl"
SUBMISSION_PATH = MODEL_DIR / "submission.csv"

print("IN_COLAB:", IN_COLAB)
print("Current working directory:", cwd)
print("DATA_DIR:", DATA_DIR.resolve())
print("TRAIN_PATH:", TRAIN_PATH.resolve())
print("TEST_PATH:", TEST_PATH.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())


IN_COLAB: True
Current working directory: /content
DATA_DIR: /content/drive/MyDrive/UD_26
TRAIN_PATH: /content/drive/MyDrive/UD_26/public_train.csv
TEST_PATH: /content/drive/MyDrive/UD_26/public_test.csv
MODEL_DIR: /content/drive/MyDrive/UD_26/model/Best_CV_BinaryCount_Stacking


In [3]:
train = pd.read_csv(TRAIN_PATH, encoding="utf-8")
test = pd.read_csv(TEST_PATH, encoding="utf-8")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("label counts:")
print(train["label"].value_counts())

assert list(train.columns) == ["row_id", "text", "label"]
assert list(test.columns) == ["row_id", "text"]
assert set(train["label"].unique()) == {"POSITIVE", "NEGATIVE"}
assert train["text"].isna().sum() == 0
assert test["text"].isna().sum() == 0

X = train["text"].astype(str)
y = train["label"].astype(str)
X_test = test["text"].astype(str)

train shape: (149995, 3)
test shape: (49997, 2)
label counts:
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64


In [4]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, mode="raw", lowercase=True):
        self.mode = mode
        self.lowercase = lowercase

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self._one(x) for x in X]

    def _base(self, x):
        x = "" if x is None else str(x)
        x = unicodedata.normalize("NFKC", html.unescape(x))
        x = re.sub(r"<[^>]+>", " ", x)
        x = re.sub(r"https?://\S+|www\.\S+", " URL ", x)
        x = re.sub(r"\s+", " ", x).strip()
        if self.lowercase:
            x = x.lower()
        return x

    def _one(self, x):
        x = self._base(x)
        if self.mode == "raw":
            return x
        if self.mode == "nospace":
            return re.sub(r"\s+", "", x)
        raise ValueError(f"unknown mode: {self.mode}")

In [5]:
BASE_CONFIG = {
    "tag": "stack_binarycount_svc0p12_0p25_0p5_ridge2_logreg3_cnb0005_meta2_rs42",
    "svc_c_low": 0.12,
    "svc_c_mid": 0.25,
    "svc_c_high": 0.50,
    "ridge_alpha": 2.0,
    "logreg_c": 3.0,
    "cnb_alpha": 0.005,
    "meta_c": 2.0,
}

SVC_LOW_VALUES = [0.08, 0.10, 0.12]
SVC_MID_VALUES = [0.23, 0.25, 0.27]
SVC_HIGH_VALUES = [0.48, 0.50, 0.52]
RIDGE_ALPHA_VALUES = [1.5, 2.0, 2.5]
LOGREG_C_VALUES = [2.0, 3.0, 4.0]
CNB_ALPHA_VALUES = [0.003, 0.005, 0.008]
META_C_VALUES = [1.5, 2.0, 2.5]

SELECTED_CONFIG = BASE_CONFIG.copy()
print("Default config:", SELECTED_CONFIG)


def config_tag(config):
    return (
        f"stack_binarycount_svc{str(config['svc_c_low']).replace('.', 'p')}_"
        f"{str(config['svc_c_mid']).replace('.', 'p')}_"
        f"{str(config['svc_c_high']).replace('.', 'p')}_"
        f"ridge{str(config['ridge_alpha']).replace('.', 'p')}_"
        f"logreg{str(config['logreg_c']).replace('.', 'p')}_"
        f"cnb{str(config['cnb_alpha']).replace('.', 'p')}_"
        f"meta{str(config['meta_c']).replace('.', 'p')}_rs42"
    )


def build_best_cv_pipeline(config=None):
    if config is None:
        config = SELECTED_CONFIG

    features = Pipeline([
        ("features", FeatureUnion([
            ("nospace_char", Pipeline([
                ("prep", TextPreprocessor(mode="nospace", lowercase=True)),
                ("vec", CountVectorizer(
                    analyzer="char",
                    ngram_range=(1, 6),
                    min_df=2,
                    max_features=800_000,
                    lowercase=False,
                    binary=True,
                    dtype=np.float32,
                )),
            ])),
            ("raw_char_wb", Pipeline([
                ("prep", TextPreprocessor(mode="raw", lowercase=True)),
                ("vec", CountVectorizer(
                    analyzer="char_wb",
                    ngram_range=(2, 6),
                    min_df=2,
                    max_features=500_000,
                    lowercase=False,
                    binary=True,
                    dtype=np.float32,
                )),
            ])),
            ("raw_word", Pipeline([
                ("prep", TextPreprocessor(mode="raw", lowercase=True)),
                ("vec", CountVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=300_000,
                    token_pattern=r"(?u)\b\w+\b",
                    lowercase=False,
                    binary=True,
                    dtype=np.float32,
                )),
            ])),
        ], n_jobs=1)),
        ("tfidf", TfidfTransformer(
            sublinear_tf=False,
            norm="l2",
            use_idf=True,
            smooth_idf=True,
        )),
    ])

    base_estimators = [
        ("svc_low", LinearSVC(C=config["svc_c_low"], random_state=RANDOM_STATE, dual="auto", max_iter=3000)),
        ("svc_mid", LinearSVC(C=config["svc_c_mid"], random_state=RANDOM_STATE, dual="auto", max_iter=3000)),
        ("svc_high", LinearSVC(C=config["svc_c_high"], random_state=RANDOM_STATE, dual="auto", max_iter=3000)),
        ("ridge", RidgeClassifier(alpha=config["ridge_alpha"])),
        ("logreg", LogisticRegression(C=config["logreg_c"], solver="liblinear", random_state=RANDOM_STATE, max_iter=1000)),
        ("cnb", ComplementNB(alpha=config["cnb_alpha"])),
    ]

    classifier = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(C=config["meta_c"], solver="liblinear", random_state=RANDOM_STATE, max_iter=1000),
        cv=3,
        stack_method="auto",
        passthrough=True,
        n_jobs=1,
    )

    return Pipeline([
        ("features", features),
        ("classifier", classifier),
    ])

pipeline = build_best_cv_pipeline()
pipeline


Default config: {'tag': 'stack_binarycount_svc0p12_0p25_0p5_ridge2_logreg3_cnb0005_meta2_rs42', 'svc_c_low': 0.12, 'svc_c_mid': 0.25, 'svc_c_high': 0.5, 'ridge_alpha': 2.0, 'logreg_c': 3.0, 'cnb_alpha': 0.005, 'meta_c': 2.0}


Pipeline(steps=[('features',
                 Pipeline(steps=[('features',
                                  FeatureUnion(n_jobs=1,
                                               transformer_list=[('nospace_char',
                                                                  Pipeline(steps=[('prep',
                                                                                   TextPreprocessor(mode='nospace')),
                                                                                  ('vec',
                                                                                   CountVectorizer(analyzer='char',
                                                                                                   binary=True,
                                                                                                   dtype=<class 'numpy.float32'>,
                                                                                                   lowercase=False,
                                                                                                   max_features=800000,
                                                                                                   min_df=2,
                                                                                                   ngram_range=(1,
                                                                                                                6)))])),
                                                                 ('raw_char_wb',
                                                                  Pipeline(steps=[(...
                                                           random_state=42)),
                                                ('svc_high',
                                                 LinearSVC(C=0.5, max_iter=3000,
                                                           random_state=42)),
                                                ('ridge',
                                                 RidgeClassifier(alpha=2.0)),
                                                ('logreg',
                                                 LogisticRegression(C=3.0,
                                                                    max_iter=1000,
                                                                    random_state=42,
                                                                    solver='liblinear')),
                                                ('cnb',
                                                 ComplementNB(alpha=0.005))],
                                    final_estimator=LogisticRegression(C=2.0,
                                                                       max_iter=1000,
                                                                       random_state=42,
                                                                       solver='liblinear'),
                                    n_jobs=1, passthrough=True))])

In [ ]:
RUN_TUNING = True


def evaluate_config(config, cv, scorer, stage):
    config = config.copy()
    config["tag"] = config_tag(config)
    scores = cross_val_score(
        build_best_cv_pipeline(config),
        X,
        y,
        scoring=scorer,
        cv=cv,
        n_jobs=1,
    )
    row = {
        "stage": stage,
        "tag": config["tag"],
        "mean_mcc": scores.mean(),
        "std_mcc": scores.std(),
        "fold_scores": scores,
        "config": config,
    }
    print(f"[{stage}] {config['tag']}: mean={scores.mean():.6f}, std={scores.std():.6f}, scores={scores}")
    return row


if RUN_TUNING:
    start = time.time()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    mcc_scorer = make_scorer(matthews_corrcoef)
    tuning_rows = []
    current_config = BASE_CONFIG.copy()

    svc_candidates = []
    for low_c in SVC_LOW_VALUES:
        for mid_c in SVC_MID_VALUES:
            for high_c in SVC_HIGH_VALUES:
                candidate = current_config.copy()
                candidate.update({"svc_c_low": low_c, "svc_c_mid": mid_c, "svc_c_high": high_c})
                svc_candidates.append(candidate)
    stage_rows = [evaluate_config(candidate, cv, mcc_scorer, "svc") for candidate in svc_candidates]
    tuning_rows.extend(stage_rows)
    current_config = max(stage_rows, key=lambda row: row["mean_mcc"])["config"].copy()
    print("Selected after svc:", current_config)

    for stage, key, values in [
        ("ridge", "ridge_alpha", RIDGE_ALPHA_VALUES),
        ("logreg", "logreg_c", LOGREG_C_VALUES),
        ("cnb", "cnb_alpha", CNB_ALPHA_VALUES),
        ("meta", "meta_c", META_C_VALUES),
    ]:
        stage_candidates = []
        for value in values:
            candidate = current_config.copy()
            candidate[key] = value
            stage_candidates.append(candidate)
        stage_rows = [evaluate_config(candidate, cv, mcc_scorer, stage) for candidate in stage_candidates]
        tuning_rows.extend(stage_rows)
        current_config = max(stage_rows, key=lambda row: row["mean_mcc"])["config"].copy()
        print(f"Selected after {stage}:", current_config)

    tuning_results = pd.DataFrame(tuning_rows).sort_values("mean_mcc", ascending=False).reset_index(drop=True)
    SELECTED_CONFIG = current_config.copy()
    SELECTED_CONFIG["tag"] = config_tag(SELECTED_CONFIG)
    print("\nTuning results by best score:")
    print(tuning_results[["stage", "tag", "mean_mcc", "std_mcc"]])
    print("Final selected config:", SELECTED_CONFIG)
    print(f"Elapsed: {time.time() - start:.1f} sec")
else:
    tuning_results = None
    SELECTED_CONFIG = BASE_CONFIG.copy()
    SELECTED_CONFIG["tag"] = config_tag(SELECTED_CONFIG)
    print("RUN_TUNING=False: using default config", SELECTED_CONFIG)

MODEL_DIR = OUTPUT_DIR / "model" / SELECTED_CONFIG["tag"]
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_MODEL_PATH = MODEL_DIR / f"{SELECTED_CONFIG['tag']}.pkl"
FINAL_MODEL_PATH = MODEL_DIR / "final_pipeline.pkl"
SUBMISSION_PATH = MODEL_DIR / "submission.csv"
print("MODEL_DIR:", MODEL_DIR.resolve())


[svc] stack_binarycount_svc0p08_0p23_0p48_ridge2p0_logreg3p0_cnb0p005_meta2p0_rs42: mean=0.766358, std=0.002961, scores=[0.76899137 0.76113944 0.76499252 0.76825779 0.76840648]


In [10]:
start = time.time()
final_pipeline = build_best_cv_pipeline(SELECTED_CONFIG)
final_pipeline.fit(X, y)

pred_test = final_pipeline.predict(X_test)
submission = pd.DataFrame({
    "row_id": test["row_id"],
    "pred_label": pred_test,
})

assert len(submission) == len(test)
assert set(submission["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}

joblib.dump(final_pipeline, EXPERIMENT_MODEL_PATH, compress=3)
joblib.dump(final_pipeline, FINAL_MODEL_PATH, compress=3)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("Saved experiment model:", EXPERIMENT_MODEL_PATH.resolve())
print("Saved final model:", FINAL_MODEL_PATH.resolve())
print("Saved submission:", SUBMISSION_PATH.resolve())
print("Prediction distribution:")
print(submission["pred_label"].value_counts())
print(f"Elapsed: {time.time() - start:.1f} sec")


Saved model: /content/drive/MyDrive/UD_26/model/Best_CV_BinaryCount_Stacking/final_pipeline.pkl
Saved submission: /content/drive/MyDrive/UD_26/model/Best_CV_BinaryCount_Stacking/submission.csv
Prediction distribution:
pred_label
POSITIVE    25044
NEGATIVE    24953
Name: count, dtype: int64
Elapsed: 430.1 sec


In [11]:
check = pd.read_csv(SUBMISSION_PATH, encoding="utf-8")
print(check.shape)
print(check.head())
print(check["pred_label"].value_counts())

assert list(check.columns) == ["row_id", "pred_label"]
assert len(check) == len(test)
assert check["row_id"].equals(test["row_id"])
assert set(check["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}
print("submission.csv format OK")

(49997, 2)
        row_id pred_label
0  test-000001   POSITIVE
1  test-000002   POSITIVE
2  test-000003   POSITIVE
3  test-000004   NEGATIVE
4  test-000005   NEGATIVE
pred_label
POSITIVE    25044
NEGATIVE    24953
Name: count, dtype: int64
submission.csv format OK
